In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

BASE_V2010 = Path('../hcris-data/HCRIS_v2010')
HCRIS_IN   = Path('../hwk4/data/output/hcris_clean.csv')
KFF_IN     = Path('data/input/kff_expansion.csv')
OUT_DIR    = Path('data/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(f'HCRIS clean:  {HCRIS_IN.resolve()}')
print(f'KFF data:     {KFF_IN.resolve()}')
print(f'Output dir:   {OUT_DIR.resolve()}')

Setup complete.
HCRIS clean:  /home/rpat638/econ470/a0/work/hwk4/data/output/hcris_clean.csv
KFF data:     /home/rpat638/econ470/a0/work/hwk5/data/input/kff_expansion.csv
Output dir:   /home/rpat638/econ470/a0/work/hwk5/data/output


In [2]:
hcris = pd.read_csv(HCRIS_IN, low_memory=False)
hcris['provider_number'] = (
    hcris['provider_number'].astype(str).str.strip().str.zfill(6)
)
hcris['year'] = hcris['year'].astype(int)

print(f'HCRIS loaded:       {len(hcris):,} rows')
print(f'Years:              {sorted(hcris["year"].unique())}')
print(f'Unique providers:   {hcris["provider_number"].nunique():,}')

HCRIS loaded:       64,259 rows
Years:              [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Unique providers:   6,860


In [3]:
MEDICARE_STATE_CODES = {
    '01': 'AL', '02': 'AK', '03': 'AZ', '04': 'AR', '05': 'CA',
    '06': 'CO', '07': 'CT', '08': 'DE', '09': 'FL', '10': 'GA',
    '11': 'HI', '12': 'ID', '13': 'IL', '14': 'IN', '15': 'IA',
    '16': 'KS', '17': 'KY', '18': 'LA', '19': 'ME', '20': 'MD',
    '21': 'MA', '22': 'MI', '23': 'MN', '24': 'MS', '25': 'MO',
    '26': 'MT', '27': 'NE', '28': 'NV', '29': 'NH', '30': 'NJ',
    '31': 'NM', '32': 'NY', '33': 'NC', '34': 'ND', '35': 'OH',
    '36': 'OK', '37': 'OR', '38': 'PA', '39': 'RI', '40': 'SC',
    '41': 'SD', '42': 'TN', '43': 'TX', '44': 'UT', '45': 'VT',
    '46': 'VA', '47': 'WA', '48': 'WV', '49': 'WI', '50': 'WY',
}

hcris['state'] = hcris['provider_number'].str[:2].map(MEDICARE_STATE_CODES)

n_before = len(hcris)
hcris = hcris.dropna(subset=['state']).copy()
print(f'Dropped non-50-state rows: {n_before - len(hcris):,}')
print(f'Remaining rows:            {len(hcris):,}')
print(f'Unique states:             {hcris["state"].nunique()}')

Dropped non-50-state rows: 3,429
Remaining rows:            60,830
Unique states:             50


In [4]:
def load_rpt(path):
    rpt = pd.read_csv(path, header=None, low_memory=False)
    rpt.columns = [
        'rpt_rec_num', 'wksht_cd', 'prvdr_num', 'npi',
        'rpt_stus_cd', 'fy_bgn_dt', 'fy_end_dt', 'proc_dt',
        'initl_rpt_sw', 'last_rpt_sw', 'trnsmtl_cd', 'fi_num',
        'adr_vndr_cd', 'fi_creat_dt', 'util_cd', 'npr_dt',
        'spec_ind', 'fi_rcpt_dt'
    ]
    rpt['rpt_rec_num'] = pd.to_numeric(rpt['rpt_rec_num'], errors='coerce')
    rpt['prvdr_num']   = rpt['prvdr_num'].astype(str).str.strip().str.zfill(6)
    rpt['fy_end_dt']   = pd.to_datetime(rpt['fy_end_dt'], errors='coerce')
    rpt['year']        = rpt['fy_end_dt'].dt.year
    return rpt[['rpt_rec_num', 'prvdr_num', 'year', 'util_cd',
                'fy_bgn_dt', 'fy_end_dt']].dropna(subset=['rpt_rec_num'])

def extract_vars(nmrc_path, rpt_df, var_map):
    targets = set()
    for ws, line, col in var_map.values():
        targets.add((ws, line, col))
    nmrc = pd.read_csv(
        nmrc_path, header=None, low_memory=False,
        names=['rpt_rec_num', 'worksheet', 'line', 'col', 'value']
    )
    nmrc['rpt_rec_num'] = pd.to_numeric(nmrc['rpt_rec_num'], errors='coerce')
    nmrc['line']        = pd.to_numeric(nmrc['line'],        errors='coerce')
    nmrc['value']       = pd.to_numeric(nmrc['value'],       errors='coerce')
    nmrc['col']         = nmrc['col'].astype(str).str.strip()
    nmrc['key']         = list(zip(nmrc['worksheet'], nmrc['line'], nmrc['col']))
    nmrc_sub            = nmrc[nmrc['key'].isin(targets)].copy()
    reverse_map         = {v: k for k, v in var_map.items()}
    nmrc_sub['var_name'] = nmrc_sub['key'].map(reverse_map)
    nmrc_sub = nmrc_sub.dropna(subset=['var_name'])
    valid_rpts = set(rpt_df['rpt_rec_num'].dropna())
    nmrc_sub   = nmrc_sub[nmrc_sub['rpt_rec_num'].isin(valid_rpts)]
    wide = (
        nmrc_sub
        .groupby(['rpt_rec_num', 'var_name'])['value']
        .first()
        .unstack('var_name')
        .reset_index()
    )
    return rpt_df.merge(wide, on='rpt_rec_num', how='left')

def get_v2010_paths(year):
    folder = BASE_V2010 / f'HospitalFY{year}'
    nmrc = next((p for p in [
        folder / f'hosp10_{year}_NMRC.CSV',
        folder / f'HOSP10_{year}_nmrc.csv',
    ] if p.exists()), None)
    rpt = next((p for p in [
        folder / f'hosp10_{year}_RPT.CSV',
        folder / f'HOSP10_{year}_rpt.csv',
    ] if p.exists()), None)
    return nmrc, rpt

VAR_MAP_S10 = {
    'uncomp_care': ('S100000', 3000, '00100'),
}

print('Functions defined.')

Functions defined.


In [5]:
uc_years = []

for yr in range(2011, 2019):
    nmrc_path, rpt_path = get_v2010_paths(yr)
    if nmrc_path is None or rpt_path is None:
        print(f'SKIP {yr} — files not found')
        continue
    print(f'Extracting S-10 for {yr}...', end=' ')
    rpt_df = load_rpt(rpt_path)
    df_yr  = extract_vars(nmrc_path, rpt_df, VAR_MAP_S10)
    uc_years.append(df_yr)
    print(f'{len(df_yr):,} reports, '
          f'{df_yr["uncomp_care"].notna().sum():,} with uncomp_care')

uc_raw = pd.concat(uc_years, ignore_index=True)
uc_raw = uc_raw.rename(columns={'prvdr_num': 'provider_number'})
print(f'\nTotal S-10 rows:      {len(uc_raw):,}')
print(f'Non-null uncomp_care: {uc_raw["uncomp_care"].notna().sum():,}')

Extracting S-10 for 2011... 6,143 reports, 5,299 with uncomp_care
Extracting S-10 for 2012... 6,202 reports, 5,196 with uncomp_care
Extracting S-10 for 2013... 6,144 reports, 5,115 with uncomp_care
Extracting S-10 for 2014... 6,190 reports, 5,140 with uncomp_care
Extracting S-10 for 2015... 6,199 reports, 5,132 with uncomp_care
Extracting S-10 for 2016... 6,204 reports, 4,935 with uncomp_care
Extracting S-10 for 2017... 6,121 reports, 4,699 with uncomp_care
Extracting S-10 for 2018... 6,159 reports, 4,712 with uncomp_care

Total S-10 rows:      49,362
Non-null uncomp_care: 40,228


In [6]:
uc_raw['year'] = uc_raw['year'].astype('Int64')
uc_raw['util_priority'] = uc_raw['util_cd'].map(
    {'F': 3, 'L': 2, 'N': 1}
).fillna(0).astype(int)

uc_dedup = (
    uc_raw
    .sort_values(
        ['provider_number', 'year', 'util_priority', 'rpt_rec_num'],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(subset=['provider_number', 'year'], keep='first')
    .drop(columns=['util_priority'])
    [['provider_number', 'year', 'uncomp_care']]
    .reset_index(drop=True)
)

uc_dedup['uncomp_care'] = uc_dedup['uncomp_care'].abs()

print(f'Deduped rows:         {len(uc_dedup):,}')
print(f'Non-null uncomp_care: {uc_dedup["uncomp_care"].notna().sum():,}')

Deduped rows:         48,660
Non-null uncomp_care: 39,676


In [7]:
hcris['year']    = hcris['year'].astype(int)
uc_dedup['year'] = uc_dedup['year'].astype(int)

hcris_uc = hcris.merge(
    uc_dedup[['provider_number', 'year', 'uncomp_care']],
    on=['provider_number', 'year'],
    how='left'
)

hcris_uc = (
    hcris_uc
    .dropna(subset=['uncomp_care'])
    .query('2011 <= year <= 2018')
    .copy()
    .reset_index(drop=True)
)

hcris_uc['uncomp_care_m'] = hcris_uc['uncomp_care'] / 1_000_000

print(f'Rows after merge & filter: {len(hcris_uc):,}')
print(f'Years:                     {sorted(hcris_uc["year"].unique())}')
print(f'Unique providers:          {hcris_uc["provider_number"].nunique():,}')
print(f'Unique states:             {hcris_uc["state"].nunique()}')

Rows after merge & filter: 35,709
Years:                     [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
Unique providers:          5,383
Unique states:             50


In [8]:
kff_raw = pd.read_csv(KFF_IN, skiprows=2)

kff = kff_raw[['Location',
               'Status of Medicaid Expansion Decision',
               'Expansion Implementation Date']].copy()
kff.columns = ['state_name', 'expansion_status', 'date_adopted']

kff = kff[kff['state_name'].notna()]
kff = kff[~kff['state_name'].str.startswith(
    ('Notes', 'Source', 'Footnote'), na=True
)]
kff = kff[kff['state_name'] != 'United States']
kff = kff[kff['state_name'] != 'District of Columbia']
kff = kff[kff['state_name'].str.len() > 2].copy()

kff['date_adopted'] = pd.to_datetime(kff['date_adopted'], errors='coerce')
kff['expand_year']  = kff['date_adopted'].dt.year
kff['expanded']     = kff['expand_year'].notna().astype(int)

print(f'KFF rows (50 states): {len(kff)}')
print(f'\nExpansion year distribution:')
print(kff['expand_year'].value_counts().sort_index())
print(f'\nNever expanded: {kff["expanded"].eq(0).sum()} states')

KFF rows (50 states): 64

Expansion year distribution:
expand_year
2,014.00    26
2,015.00     3
2,016.00     2
2,019.00     2
2,020.00     3
2,021.00     2
2,023.00     2
Name: count, dtype: int64

Never expanded: 24 states


In [10]:
xwalk = pd.DataFrame({
    'state': [
        'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
        'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
        'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
        'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
        'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
    ],
    'state_name': [
        'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
        'Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho',
        'Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana',
        'Maine','Maryland','Massachusetts','Michigan','Minnesota',
        'Mississippi','Missouri','Montana','Nebraska','Nevada',
        'New Hampshire','New Jersey','New Mexico','New York',
        'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
        'Pennsylvania','Rhode Island','South Carolina','South Dakota',
        'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
        'West Virginia','Wisconsin','Wyoming'
    ]
})

print(f'Crosswalk rows: {len(xwalk)}')

Crosswalk rows: 50


In [12]:
hcris_named = hcris_uc.merge(xwalk, on='state', how='left')

merged = hcris_named.merge(
    kff[['state_name', 'expand_year', 'expanded', 'date_adopted']],
    on='state_name',
    how='left'
)

print(f'Merged shape:        {merged.shape}')
print(f'Unique states:       {merged["state"].nunique()}')
print(f'Missing expand_year: {merged["expand_year"].isna().sum():,} '
      f'(never-expanders )')

Merged shape:        (35709, 29)
Unique states:       50
Missing expand_year: 6,762 (never-expanders )


In [13]:
df = merged.copy()
df['year'] = df['year'].astype(int)

df['expand_ever'] = df['expanded'].fillna(0).astype(int)

df['treat'] = (
    df['expand_ever'].eq(1) &
    df['year'].ge(df['expand_year'].fillna(9999).astype(int))
).astype(int)

df['post'] = (df['year'] >= 2014).astype(int)

df['time_to_treat'] = np.where(
    df['expand_ever'] == 1,
    df['year'] - df['expand_year'].fillna(0).astype(int),
    0
)

df['time_to_treat_bin'] = (
    df['time_to_treat'].clip(lower=-3, upper=5).astype(int)
)

print('Variables created:')
print(df[['expand_ever', 'treat', 'post',
          'time_to_treat', 'time_to_treat_bin']].describe().round(2))

Variables created:
       expand_ever     treat      post  time_to_treat  time_to_treat_bin
count    35,709.00 35,709.00 35,709.00      35,709.00          35,709.00
mean          0.81      0.38      0.65          -0.94              -0.28
std           0.39      0.49      0.48           3.33               2.16
min           0.00      0.00      0.00         -12.00              -3.00
25%           1.00      0.00      0.00          -2.00              -2.00
50%           1.00      0.00      1.00           0.00               0.00
75%           1.00      1.00      1.00           1.00               1.00
max           1.00      1.00      1.00           4.00               4.00


In [14]:
sample_A = df.copy()

sample_B = df[
    df['expand_year'].isna() |
    df['expand_year'].eq(2014)
].copy().reset_index(drop=True)

sample_B_2x2 = sample_B[
    sample_B['year'].isin([2012, 2015])
].copy().reset_index(drop=True)

print(f'Sample A (full panel):       {len(sample_A):,} rows | '
      f'{sample_A["provider_number"].nunique():,} hospitals | '
      f'{sample_A["state"].nunique()} states')
print(f'Sample B (2014 + never):     {len(sample_B):,} rows | '
      f'{sample_B["provider_number"].nunique():,} hospitals | '
      f'{sample_B["state"].nunique()} states')
print(f'  2014 expander states:      '
      f'{sample_B[sample_B["expand_year"]==2014]["state"].nunique()}')
print(f'  Never-expander states:     '
      f'{sample_B[sample_B["expand_year"].isna()]["state"].nunique()}')
print(f'Sample B 2x2 (2012 + 2015): {len(sample_B_2x2):,} rows | '
      f'{sample_B_2x2["provider_number"].nunique():,} hospitals')

Sample A (full panel):       35,709 rows | 5,383 hospitals | 50 states
Sample B (2014 + never):     25,391 rows | 3,830 hospitals | 36 states
  2014 expander states:      26
  Never-expander states:     10
Sample B 2x2 (2012 + 2015): 6,868 rows | 3,619 hospitals


In [24]:
keep_cols = [
    'provider_number', 'year', 'state', 'state_name',
    'uncomp_care', 'uncomp_care_m',
    'expand_year', 'expand_ever', 'expanded', 'date_adopted',
    'post', 'treat', 'time_to_treat', 'time_to_treat_bin'
]

sample_A[keep_cols].to_csv(OUT_DIR / 'hcris_kff_full.csv', index=False)
sample_B[keep_cols].to_csv(OUT_DIR / 'hcris_kff_2014.csv', index=False)

print('=== OUTPUT FILES ===')
for fname, desc in [
    ('hcris_kff_full.csv', 'Full panel'),
    ('hcris_kff_2014.csv', '2014 + never expanders'),
]:
    fpath = OUT_DIR / fname
    tmp   = pd.read_csv(fpath)
    print(f'  ✅  {fname:<30s}  {len(tmp):>7,} rows   {desc}')

=== OUTPUT FILES ===
  ✅  hcris_kff_full.csv               35,709 rows   Full panel
  ✅  hcris_kff_2014.csv               25,391 rows   2014 + never expanders


In [26]:
print('=== COLUMN CHECK ===')
for col in keep_cols:
    ok = col in sample_A.columns
    print(f'  {"✓" if ok else "✗  MISSING"}  {col}')

print('\n=== YEAR COVERAGE ===')
print(f'Full panel years:       {sorted(sample_A["year"].unique())}')
print(f'2014 sample years:      {sorted(sample_B["year"].unique())}')
print(f'2x2 sample years:       {sorted(sample_B_2x2["year"].unique())}')
print('\nNote: Missing 2010 S-10 data')
print('for fiscal year 2010 in HCRIS v2010 files.')

print('\n=== EXPANSION STATUS CHECK ===')
chk = (
    sample_A[['state', 'expand_year', 'expand_ever']]
    .drop_duplicates('state')
    .sort_values('expand_year')
)
print(chk.to_string(index=False))

=== COLUMN CHECK ===
  ✓  provider_number
  ✓  year
  ✓  state
  ✓  state_name
  ✓  uncomp_care
  ✓  uncomp_care_m
  ✓  expand_year
  ✓  expand_ever
  ✓  expanded
  ✓  date_adopted
  ✓  post
  ✓  treat
  ✓  time_to_treat
  ✓  time_to_treat_bin

=== YEAR COVERAGE ===
Full panel years:       [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
2014 sample years:      [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]
2x2 sample years:       [np.int64(2012), np.int64(2015)]

Note: Missing 2010 S-10 data
for fiscal year 2010 in HCRIS v2010 files.

=== EXPANSION STATUS CHECK ===
state  expand_year  expand_ever
   AZ     2,014.00            1
   AR     2,014.00            1
   CA     2,014.00            1
   CO     2,014.00            1
   DE     2,014.00            1
   CT     2,014.00            1
   HI     2,014.00            1
   IL    